<a href="https://colab.research.google.com/github/Zikai-Dou/dataflow-ai-labs/blob/main/Lab6_Roofline_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6: Roofline Analysis for LLM Inference

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Module**: 6, Performance Modeling  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes


---

## Learning objectives
By the end of this lab you will be able to:
1. Compute arithmetic intensity for common operators from first principles.
2. Construct an empirical roofline for your GPU from measured peaks.
3. Profile real LLM kernels and place them on the roofline.
4. Compare GPU and IBM Spyre with a dual roofline and a capacity-aware crossover.

## Prerequisites
Labs 1-5 complete, plus the Module 6 lecture on the roofline model, arithmetic intensity, and the ridge point.

## Working through this lab
1. Run the notebook cells in order.
2. Complete each learner task before opening its **Hint**.
3. If you get stuck, expand the **Hint** to view the full reference solution from the authoring notebook.
4. Run each **Self-check** cell after completing the corresponding task.



## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                          ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## Setup
Run the cell below to install dependencies and detect your GPU.

In [ ]:
# Shared hardware reference. identical across every lab in this course.
# ridge = peak FP16 TFLOPS / peak memory bandwidth (TB/s): the arithmetic intensity
# (FLOPs/byte) where a workload flips from memory-bound to compute-bound. These match
# the lecture decks (T4 ridge ~203, H100 ~295).
HW_SPECS = {
    'Tesla T4':              {'fp16_tflops': 65.0,  'bw_tb_s': 0.320, 'ridge': 203},
    'NVIDIA A100-SXM4-40GB': {'fp16_tflops': 312.0, 'bw_tb_s': 1.555, 'ridge': 201},
    'NVIDIA A100-SXM4-80GB': {'fp16_tflops': 312.0, 'bw_tb_s': 2.039, 'ridge': 153},
    'NVIDIA H100':           {'fp16_tflops': 989.0, 'bw_tb_s': 3.35,  'ridge': 295},
}
# IBM Spyre (confirm against the current spec sheet before publishing):
SPYRE_SPECS = {'fp16_tflops': 98.0, 'lpddr5_tb_s': 0.204, 'scratchpad_mb': 64, 'scratchpad_tb_s': 10.0}

def lookup_hw(name):
    for _k, _v in HW_SPECS.items():
        if _k in name:
            return _k, _v
    return 'Tesla T4', HW_SPECS['Tesla T4']   # Colab free-tier default


> **Roofline & Arithmetic-Intensity Conventions (this notebook)**
> - **Precision:** FP16 (IEEE half) unless otherwise noted
> - **FLOPs counting:** 1 fused multiply-add = 2 FLOPs
> - **Memory level:** Device DRAM (HBM/GDDR), not L2 or scratchpad
> - **Byte convention:** Bytes *transferred* between DRAM and compute (reads + writes)
> - **Sparsity:** Dense (no structural sparsity) unless stated
> - **Hardware SKU:** Detected at runtime via `torch.cuda.get_device_name()`; specs from `hardware_profiles.json`

In [ ]:
!pip install -q transformers==4.44.2 matplotlib numpy

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.profiler import profile, ProfilerActivity, record_function
import time

# Detect GPU
assert torch.cuda.is_available(), "This lab requires a GPU runtime. Go to Runtime > Change runtime type > GPU."
device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")

_hwname, _hw = lookup_hw(gpu_name)
PEAK_TFLOPS = _hw['fp16_tflops']; PEAK_BW_TB = _hw['bw_tb_s']; RIDGE_POINT = _hw['ridge']
if _hwname not in gpu_name:
    print(f"Note: unknown GPU '{gpu_name}'. Using {_hwname} specs as default.")

print(f"\nHardware Specs:")
print(f"  Peak FP16 Compute: {PEAK_TFLOPS} TFLOPS")
print(f"  Peak Memory BW:    {PEAK_BW_TB} TB/s")
print(f"  Ridge Point:       {RIDGE_POINT} FLOPs/byte")

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


---
## Part 1: Arithmetic Intensity from First Principles

Calculate the arithmetic intensity of three common DNN operators.

**Recall**: Arithmetic Intensity (AI) = FLOPs / Bytes transferred

For matrix multiply C = A × B:
- FLOPs = 2 × M × N × K (multiply-add for each output element)
- Bytes = (M×K + K×N + M×N) × bytes_per_element (read A, read B, write C)

In [ ]:
# Arithmetic intensity for each operator (FP16 = 2 bytes/element)
# --- Operator 1: Large GEMM (Prefill FFN) ---
# --- Operator 2: Small GEMM (Decode, batch=1) ---

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Arithmetic intensity for each operator (FP16 = 2 bytes/element)
BYTES_PER_ELEM = 2

# --- Operator 1: Large GEMM (Prefill FFN) ---
M1, K1, N1 = 2048, 4096, 16384
flops_gemm_large = 2 * M1 * K1 * N1
bytes_gemm_large = (M1*K1 + K1*N1 + M1*N1) * BYTES_PER_ELEM
ai_gemm_large    = flops_gemm_large / bytes_gemm_large
print(f"Operator 1 - Large GEMM (Prefill FFN):")
print(f"  Shape: [{M1},{K1}] x [{K1},{N1}]")
print(f"  FLOPs: {flops_gemm_large:,.0f}")
print(f"  Bytes: {bytes_gemm_large:,.0f}")
print(f"  AI:    {ai_gemm_large:.1f} FLOPs/byte")
print()

# --- Operator 2: Small GEMM (Decode, batch=1) ---
M2, K2, N2 = 1, 4096, 16384
flops_gemm_small = 2 * M2 * K2 * N2
bytes_gemm_small = (M2*K2 + K2*N2 + M2*N2) * BYTES_PER_ELEM
ai_gemm_small    = flops_gemm_small / bytes_gemm_small
print(f"Operator 2 - Small GEMM (Decode FFN, batch=1):")
print(f"  Shape: [{M2},{K2}] x [{K2},{N2}]")
print(f"  FLOPs: {flops_gemm_small:,.0f}")
print(f"  Bytes: {bytes_gemm_small:,.0f}")
print(f"  AI:    {ai_gemm_small:.2f} FLOPs/byte")
print()

# --- Operator 3: Attention QK^T (Decode, single head) ---
seq_len, head_dim = 2048, 128
flops_attn = 2 * 1 * seq_len * head_dim
bytes_attn = (1*head_dim + seq_len*head_dim + 1*seq_len) * BYTES_PER_ELEM
ai_attn    = flops_attn / bytes_attn
print(f"Operator 3 - Attention QK^T (Decode, single head):")
print(f"  Q shape: [1,{head_dim}], K shape: [{seq_len},{head_dim}]")
print(f"  FLOPs: {flops_attn:,.0f}")
print(f"  Bytes: {bytes_attn:,.0f}")
print(f"  AI:    {ai_attn:.2f} FLOPs/byte")
```

</details>


In [ ]:
# --- Self-check 1 ---
assert ai_gemm_large is not None, "Complete the large GEMM AI calculation"
assert ai_gemm_small is not None, "Complete the small GEMM AI calculation"
assert ai_attn is not None, "Complete the attention AI calculation"

# Verify ranges (not exact values; allow for rounding)
assert 100 < ai_gemm_large < 5000, f"Large GEMM AI={ai_gemm_large:.1f} seems wrong. Expected ~200-2000."
assert 0.5 < ai_gemm_small < 5, f"Small GEMM AI={ai_gemm_small:.2f} seems wrong. Expected ~1."
assert 0.1 < ai_attn < 3, f"Attention AI={ai_attn:.2f} seems wrong. Expected ~0.5-1."

print("[OK] Self-check 1 passed!")
print(f"\nKey insight: Prefill GEMM AI ({ai_gemm_large:.0f}) vs Decode GEMM AI ({ai_gemm_small:.2f})")
print(f"  → {ai_gemm_large/ai_gemm_small:.0f}× difference from batch size alone!")
print(f"  → Prefill is {'COMPUTE' if ai_gemm_large > RIDGE_POINT else 'MEMORY'}-bound on this GPU")
print(f"  → Decode is {'COMPUTE' if ai_gemm_small > RIDGE_POINT else 'MEMORY'}-bound on this GPU")

---
## Part 2: Empirical Roofline Construction

Measure the GPU's actual achievable peak compute and memory bandwidth, then plot the roofline.

We'll use:
- A large matrix multiply to measure peak compute
- A memory copy to measure peak bandwidth

### Measurement Best Practices

> **⚠️ Single-run timings can be misleading.** GPU performance varies due to thermal throttling,
> background processes, and memory state. Always use multiple measurements.

**Protocol for reliable benchmarks:**
1. **Warmup:** 3-5 iterations (discarded) to fill caches and trigger JIT compilation
2. **Repeat:** N ≥ 10 measurement iterations
3. **Report:** Median (robust to outliers) ± IQR or percentiles (P5, P50, P95)
4. **Synchronize:** Always call `torch.cuda.synchronize()` before timing GPU ops
5. **Isolate:** Use `torch.inference_mode()` or `torch.no_grad()` to avoid autograd overhead


In [ ]:
# Utility: robust timing with statistics
import time
import numpy as np

def bench_robust(fn, n_warmup=5, n_measure=20, sync_cuda=True):
    """Run fn() with warmup, return timing statistics in milliseconds."""
    # Warmup
    for _ in range(n_warmup):
        fn()
    if sync_cuda and torch.cuda.is_available():
        torch.cuda.synchronize()

    # Measure
    times = []
    for _ in range(n_measure):
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    return {
        "median_ms": float(np.median(times)),
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "p5_ms": float(np.percentile(times, 5)),
        "p95_ms": float(np.percentile(times, 95)),
        "iqr_ms": float(np.percentile(times, 75) - np.percentile(times, 25)),
        "n": n_measure,
    }

# Example usage
if torch.cuda.is_available():
    x = torch.randn(1024, 1024, device=DEVICE)
    stats = bench_robust(lambda: torch.mm(x, x))
    print(f"Matrix multiply (1024x1024) on {DEVICE}:")
    print(f"  Median: {stats['median_ms']:.3f} ms")
    print(f"  P5-P95: [{stats['p5_ms']:.3f}, {stats['p95_ms']:.3f}] ms")
    print(f"  IQR:    {stats['iqr_ms']:.3f} ms (over {stats['n']} runs)")
else:
    print("Skipping bench_robust demo (no GPU)")


In [ ]:
    # Warmup
    # Measure
    # Warmup

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
def measure_peak_compute(size=4096, dtype=torch.float16, warmup=5, trials=20):
    """Measure achieved compute via large GEMM (high AI → compute-bound)."""
    A = torch.randn(size, size, dtype=dtype, device=device)
    B = torch.randn(size, size, dtype=dtype, device=device)
    
    # Warmup
    for _ in range(warmup):
        torch.mm(A, B)
    torch.cuda.synchronize()
    
    # Measure
    times = []
    for _ in range(trials):
        torch.cuda.synchronize()
        start = time.perf_counter()
        torch.mm(A, B)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    
    min_time = min(times)
    flops = 2 * size * size * size
    achieved_tflops = flops / min_time / 1e12
    return achieved_tflops


def measure_peak_bandwidth(size_mb=256, dtype=torch.float16, warmup=5, trials=20):
    """Measure achieved bandwidth via large memory copy."""
    num_elements = (size_mb * 1024 * 1024) // 2  # FP16 = 2 bytes
    src = torch.randn(num_elements, dtype=dtype, device=device)
    dst = torch.empty_like(src)
    
    # Warmup
    for _ in range(warmup):
        dst.copy_(src)
    torch.cuda.synchronize()
    
    # Measure
    times = []
    for _ in range(trials):
        torch.cuda.synchronize()
        start = time.perf_counter()
        dst.copy_(src)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    
    min_time = min(times)
    total_bytes = 2 * num_elements * 2  # read + write
    achieved_bw_tb = total_bytes / min_time / 1e12
    return achieved_bw_tb


print("Measuring achieved peak compute (large GEMM)...")
achieved_compute = measure_peak_compute()
print(f"  Achieved: {achieved_compute:.1f} TFLOPS ({100*achieved_compute/PEAK_TFLOPS:.0f}% of spec)")

print("\nMeasuring achieved peak bandwidth (memory copy)...")
achieved_bw = measure_peak_bandwidth()
print(f"  Achieved: {achieved_bw:.3f} TB/s ({100*achieved_bw/PEAK_BW_TB:.0f}% of spec)")

achieved_ridge = achieved_compute / achieved_bw
print(f"\nEmpirical ridge point: {achieved_ridge:.0f} FLOPs/byte")
print(f"Spec ridge point:      {RIDGE_POINT} FLOPs/byte")
```

</details>


In [ ]:
# TODO: Plot the roofline model
# X-axis: Arithmetic Intensity (log scale, range 0.1 to 10000)
# Y-axis: Performance in TFLOPS (log scale)

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# TODO: Plot the roofline model
# X-axis: Arithmetic Intensity (log scale, range 0.1 to 10000)
# Y-axis: Performance in TFLOPS (log scale)
# Plot: (1) bandwidth diagonal, (2) compute horizontal, (3) your 3 operators from Section 1

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Arithmetic intensity range (log scale)
ai_range = np.logspace(-1, 4, 1000)  # 0.1 to 10000

# TODO: Calculate roofline performance at each AI value
# roofline_perf = min(achieved_compute, achieved_bw * ai) for each ai value
roofline_perf = np.minimum(achieved_compute, achieved_bw * ai_range)  # BW[TB/s] * AI[FLOP/byte] = TFLOP/s
# Hint: achieved_bw is in TB/s, so BW * AI gives TFLOPS directly when AI is in FLOP/byte and BW is in TB/s

# Plot the roofline
ax.loglog(ai_range, roofline_perf, 'b-', linewidth=2, label='GPU Roofline')

# TODO: Plot the three operators from Section 1 as colored dots
# For each operator, its achieved performance is min(peak_compute, peak_bw * ai)
operators = {
    "Prefill FFN GEMM": ai_gemm_large,
    "Decode FFN GEMM": ai_gemm_small,
    "Decode Attention QK^T": ai_attn,
}

colors = ['green', 'red', 'orange']
for (name, ai), color in zip(operators.items(), colors):
    perf = min(achieved_compute, achieved_bw * ai)
    ax.plot(ai, perf, 'o', markersize=12, color=color, label=f"{name} (AI={ai:.1f})")

# Mark the ridge point
ax.axvline(x=achieved_ridge, color='gray', linestyle='--', alpha=0.5, label=f'Ridge = {achieved_ridge:.0f}')

ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOPS)', fontsize=12)
ax.set_title(f'Roofline Model; {gpu_name}', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 10000)
ax.set_ylim(0.01, PEAK_TFLOPS * 2)

plt.tight_layout()
plt.savefig('roofline_gpu.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roofline_gpu.png")
```

</details>


In [ ]:
# --- Self-check 2 ---
assert roofline_perf is not None, "Complete the roofline_perf calculation"
assert achieved_compute > 0.25 * PEAK_TFLOPS, f"Achieved compute ({achieved_compute:.1f}) seems too low. Check GPU runtime."
assert achieved_bw > 0.3 * PEAK_BW_TB, f"Achieved BW ({achieved_bw:.3f}) seems too low."

print("[OK] Self-check 2 passed!")
print(f"  Your GPU achieves {100*achieved_compute/PEAK_TFLOPS:.0f}% of spec compute and {100*achieved_bw/PEAK_BW_TB:.0f}% of spec bandwidth.")

---
## Part 3: LLM Kernel Profiling

Profile a real LLM running inference and extract per-kernel performance data.
We'll use GPT-2 (small enough for Colab) to demonstrate the pattern.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"  # 124M params; fits easily on T4
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
model.eval()

# Model specs
hidden = model.config.n_embd        # 768
n_layers = model.config.n_layer     # 12
n_heads = model.config.n_head       # 12
head_dim = hidden // n_heads        # 64
ffn_hidden = 4 * hidden             # 3072

print(f"Model: {model_name}")
print(f"  Hidden dim: {hidden}")
print(f"  Layers: {n_layers}")
print(f"  Heads: {n_heads}, Head dim: {head_dim}")
print(f"  FFN hidden: {ffn_hidden}")
print(f"  Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Profile prefill (all tokens processed at once)
prompt = "The roofline model is a visual performance analysis tool that helps engineers"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
seq_len_input = inputs["input_ids"].shape[1]
print(f"Input sequence length: {seq_len_input} tokens")

# Profile with FLOP counting
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    with_flops=True,
    profile_memory=True,
) as prof:
    with torch.no_grad():
        outputs = model(**inputs)

# Extract key events
events = prof.key_averages()
print(f"\nTop operators by CUDA time:")
sort_key = "cuda_time_total" if any(getattr(e, "cuda_time_total", 0) > 0 for e in events) else "cpu_time_total"
print(events.table(sort_by=sort_key, row_limit=15, top_level_events_only=False))

In [ ]:
# TODO: Extract arithmetic intensity data from profiler
# For each operator with FLOPs > 0, compute:
#   - FLOPs (from profiler)

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# TODO: Extract arithmetic intensity data from profiler
# For each operator with FLOPs > 0, compute:
#   - FLOPs (from profiler)
#   - Time (CUDA time)
#   - Achieved TFLOPS = FLOPs / time
#   - Estimated AI (from shape analysis)

kernel_data = []  # list of dicts: {name, flops, time_us, achieved_tflops, estimated_ai}

for event in events:
    time_us = getattr(event, "cuda_time_total", 0) or event.self_cpu_time_total
    if event.flops and event.flops > 0 and time_us > 0:
        flops = event.flops
        time_s = time_us / 1e6
        achieved_tflops = flops / time_s / 1e12
        
        # TODO: Estimate bytes transferred for this op
        # Approximation: for GEMM [M,K]x[K,N], bytes ≈ (M*K + K*N + M*N) * 2
        # For ops where we can't determine shape, use: bytes ≈ flops / ridge_point
        # (This gives the "expected" bytes if the op were at the ridge)
        
        # Simple heuristic: use achieved performance to infer actual AI
        # If achieved << peak_compute, then op is memory-bound
        # estimated_ai ≈ achieved_tflops / achieved_bw
        estimated_ai = achieved_tflops / achieved_bw if achieved_bw > 0 else 0
        
        kernel_data.append({
            "name": event.key[:40],
            "flops": flops,
            "time_us": time_us,
            "achieved_tflops": achieved_tflops,
            "estimated_ai": estimated_ai,
        })

# Sort by time (most expensive first)
kernel_data.sort(key=lambda x: x["time_us"], reverse=True)

print(f"\nExtracted {len(kernel_data)} kernels with FLOP data:")
print(f"{'Kernel':<42} {'FLOPs':>12} {'Time(μs)':>10} {'TFLOPS':>8} {'Est.AI':>8}")
print("-" * 85)
for k in kernel_data[:10]:
    print(f"{k['name']:<42} {k['flops']:>12,.0f} {k['time_us']:>10.0f} {k['achieved_tflops']:>8.2f} {k['estimated_ai']:>8.1f}")
```

</details>


In [ ]:
# --- Self-check 3 ---
assert len(kernel_data) > 0, "No kernel data extracted. Make sure profiling ran correctly."
assert any(k["achieved_tflops"] > 0 for k in kernel_data), "No kernels achieved measurable TFLOPS."

print("[OK] Self-check 3 passed!")
print(f"  Profiled {len(kernel_data)} compute kernels.")
max_kernel = max(kernel_data, key=lambda x: x["achieved_tflops"])
print(f"  Highest achiever: {max_kernel['name']} at {max_kernel['achieved_tflops']:.2f} TFLOPS")

---
## Part 4: Roofline Diagnosis

Plot profiled kernels on the roofline and classify each as compute-bound or memory-bound.
Then propose one optimization for the most time-consuming memory-bound kernel.

In [ ]:
# TODO: Plot profiled kernels on the roofline from Section 2
# Plot roofline
# TODO: Plot each profiled kernel as a dot

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# TODO: Plot profiled kernels on the roofline from Section 2

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot roofline
ai_range = np.logspace(-1, 4, 1000)
roofline = np.minimum(achieved_compute, achieved_bw * ai_range)
ax.loglog(ai_range, roofline, 'b-', linewidth=2, label='Roofline', alpha=0.7)

# TODO: Plot each profiled kernel as a dot
# X = estimated_ai, Y = achieved_tflops
# Color by classification: green=compute-bound (near ceiling), red=memory-bound (below ceiling)

for k in kernel_data[:15]:  # Top 15 by time
    ai = k["estimated_ai"]
    perf = k["achieved_tflops"]
    if ai > 0 and perf > 0:
        # Classify: if achieved > 50% of roofline at this AI, it's well-optimized
        roof_at_ai = min(achieved_compute, achieved_bw * ai)
        efficiency = perf / roof_at_ai if roof_at_ai > 0 else 0
        color = 'green' if efficiency > 0.5 else 'red'
        ax.plot(ai, perf, 'o', color=color, markersize=8, alpha=0.7)
        ax.annotate(k["name"][:15], (ai, perf), fontsize=7, alpha=0.8)

ax.axvline(x=achieved_ridge, color='gray', linestyle='--', alpha=0.5, label=f'Ridge={achieved_ridge:.0f}')
ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
ax.set_ylabel('Achieved Performance (TFLOPS)', fontsize=12)
ax.set_title(f'LLM Kernel Roofline; {model_name} on {gpu_name}', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 10000)

plt.tight_layout()
plt.savefig('roofline_kernels.png', dpi=150, bbox_inches='tight')
plt.show()
```

</details>


In [ ]:
# Classification table for the top-5 kernels by time

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Classification table for the top-5 kernels by time
print("Kernel Classification Table:")
print(f"{'Kernel':<35} {'AI':>6} {'Achieved':>9} {'Ceiling':>9} {'Efficiency':>10} {'Bound':>10}")
print("-" * 85)
for k in kernel_data[:5]:
    ai = k["estimated_ai"]
    perf = k["achieved_tflops"]
    ceiling = min(achieved_compute, achieved_bw * ai) if ai > 0 else achieved_compute
    efficiency = perf / ceiling if ceiling > 0 else 0
    bound = "COMPUTE" if ai > achieved_ridge else "MEMORY"
    print(f"{k['name'][:35]:<35} {ai:>6.1f} {perf:>8.2f}T {ceiling:>8.2f}T {efficiency:>9.0%} {bound:>10}")

optimization_proposal = """
The most time-consuming kernels are the FFN / projection GEMMs (aten::addmm / mm) and,
at longer context, the attention matmuls. At batch=1 decode these sit far LEFT of the
ridge (~217 FLOP/byte on a T4), so they are MEMORY-bound: the kernel spends its time
reading weights from GDDR, not doing math.

Best single optimization: QUANTIZATION. Casting weights FP16 -> INT8 (or INT4) halves
(quarters) the bytes moved per MAC. On the roofline, a memory-bound point's attainable
performance is bandwidth x arithmetic-intensity, so halving the bytes DOUBLES the
operator's arithmetic intensity and slides it right and up the bandwidth diagonal,
toward the ridge. It does not raise the roof; it moves the operator along it. (Fusion
helps the elementwise/norm ops, not these weight-bound GEMMs; larger batch helps but
changes single-stream decode semantics.)
"""
print(f"\nOptimization Proposal:\n{optimization_proposal}")
```

</details>


In [ ]:
# --- Self-check 4 ---
assert "TODO" not in optimization_proposal, "Please write your optimization proposal (replace the TODO)."
assert len(optimization_proposal.strip()) > 50, "Optimization proposal should be at least a few sentences."

print("[OK] Self-check 4 passed!")
print("  Classification and optimization proposal recorded.")

---
## Part 5: Dual Roofline: GPU vs. Spyre Crossover

Construct a dual roofline comparing your GPU with IBM Spyre's scratchpad-based architecture.
Identify which operators from Section 3 would benefit from running on Spyre.

**Spyre specs**:
- Peak FP16 compute: 98 TFLOPS
- External memory (LPDDR5): 204 GB/s
- On-chip scratchpad: 64 MB, effective bandwidth >> 204 GB/s
- For this exercise, assume scratchpad effective bandwidth of **10 TB/s** (conservative estimate for ring interconnect across 32 cores)

In [ ]:
# Spyre specs
# Ridge points
# TODO: Plot dual roofline

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Spyre specs
SPYRE_PEAK_TFLOPS = 98.0
SPYRE_LPDDR5_BW = 0.204  # TB/s
SPYRE_SCRATCHPAD_BW = 10.0  # TB/s (conservative estimate for on-chip)
SPYRE_SCRATCHPAD_SIZE_MB = 64

# Ridge points
spyre_ridge_lpddr5 = SPYRE_PEAK_TFLOPS / SPYRE_LPDDR5_BW
spyre_ridge_scratchpad = SPYRE_PEAK_TFLOPS / SPYRE_SCRATCHPAD_BW

print(f"Spyre Ridge Points:")
print(f"  LPDDR5 level:     {spyre_ridge_lpddr5:.0f} FLOPs/byte")
print(f"  Scratchpad level: {spyre_ridge_scratchpad:.0f} FLOPs/byte")
print(f"  GPU ridge:        {achieved_ridge:.0f} FLOPs/byte")

# TODO: Plot dual roofline
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

ai_range = np.logspace(-1, 4, 1000)

# GPU roofline (your measured GPU)
gpu_roof = np.minimum(achieved_compute, achieved_bw * ai_range)
ax.loglog(ai_range, gpu_roof, 'b-', linewidth=2.5, label=f'GPU ({gpu_name})')

# TODO: Spyre roofline; scratchpad level (for data that fits in 64MB)
spyre_scratchpad_roof = np.minimum(SPYRE_PEAK_TFLOPS, SPYRE_SCRATCHPAD_BW * ai_range)

# TODO: Spyre roofline; LPDDR5 level (for data that doesn't fit)
spyre_lpddr5_roof = np.minimum(SPYRE_PEAK_TFLOPS, SPYRE_LPDDR5_BW * ai_range)

ax.loglog(ai_range, spyre_scratchpad_roof, 'g-', linewidth=2.5,
          label=f'Spyre (scratchpad, {SPYRE_SCRATCHPAD_SIZE_MB}MB on-chip)')
ax.loglog(ai_range, spyre_lpddr5_roof, 'g--', linewidth=1.5, alpha=0.6,
          label='Spyre (LPDDR5 fallback)')

# Find crossover point (where Spyre scratchpad > GPU)
# TODO: Find the AI value where spyre_scratchpad_roof > gpu_roof
crossover_mask = spyre_scratchpad_roof > gpu_roof
if crossover_mask.any():
    crossover_ai = ai_range[crossover_mask][-1]  # highest AI where Spyre wins
    ax.axvline(x=crossover_ai, color='purple', linestyle=':', alpha=0.7,
              label=f'Crossover AI ≈ {crossover_ai:.1f}')
    ax.fill_between(ai_range, 0.001, 1000, where=crossover_mask,
                   alpha=0.05, color='green')

# Plot decode operators from Section 1
ax.plot(ai_gemm_small, min(achieved_compute, achieved_bw * ai_gemm_small),
       's', markersize=14, color='red', label=f'Decode FFN (AI={ai_gemm_small:.1f})', zorder=5)
ax.plot(ai_attn, min(achieved_compute, achieved_bw * ai_attn),
       '^', markersize=14, color='orange', label=f'Decode Attn (AI={ai_attn:.2f})', zorder=5)

ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOPS)', fontsize=12)
ax.set_title('Dual Roofline: GPU vs. IBM Spyre', fontsize=14)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 10000)
ax.set_ylim(0.01, max(PEAK_TFLOPS, SPYRE_PEAK_TFLOPS) * 2)

plt.tight_layout()
plt.savefig('roofline_dual.png', dpi=150, bbox_inches='tight')
plt.show()
```

</details>


In [ ]:
dual_roofline_analysis = """
YOUR ANALYSIS HERE
"""
print(dual_roofline_analysis)


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Reference written analysis
dual_roofline_analysis = """
1. Crossover. With scratchpad BW = 10 TB/s vs the T4's ~0.3 TB/s, and Spyre peak 98 vs
   the T4's achieved ~50 TFLOPS, the Spyre *scratchpad* roofline sits above the GPU
   roofline across essentially the whole AI range (the reported crossover_ai lands near
   the top of the sweep). Honest reading: when the working set fits on-chip, Spyre beats
   the GPU at every AI here -- there is no interior crossover. The real crossover is a
   CAPACITY threshold (does the data fit in 64 MB?), not an AI value.

2. The decode operators (Decode FFN GEMM AI ~= 1, Decode Attention AI ~= 1) sit deep in
   the memory-bound region, which is exactly where a 10 TB/s scratchpad beats 0.3 TB/s
   GDDR by ~30x. Those benefit most from Spyre -- if they fit on-chip.

3. Decode FFN at AI ~= 1: scratchpad-bound perf ~ 10 TB/s x 1 = 10 TFLOP/s vs the T4's
   ~0.3 TB/s x 1 = 0.3 TFLOP/s -- roughly a 30x bandwidth-limited advantage before either
   peak is reached.

4. A 7B model is 14 GB in FP16; the scratchpad is 64 MB. The weights do NOT fit, so Spyre
   streams from LPDDR5 (0.204 TB/s), which is BELOW the T4's GDDR bandwidth. On the LPDDR5
   roofline those memory-bound decode ops are actually slower than on the GPU. The
   scratchpad advantage is capacity-gated: it applies only to the working set that fits.

5. Per-layer fit: 64 MB / 2 bytes = 32M parameters can live in the scratchpad at once. So
   the advantage applies when a SINGLE layer's weights (plus its live activation / KV tile)
   fit in 64 MB, and you tile/stream layer by layer. That is why the real Spyre story is
   explicit compiler-managed staging, not "the whole model lives in SRAM."
"""
print(dual_roofline_analysis)
```

</details>


In [ ]:
# --- Self-check 5 ---
assert spyre_scratchpad_roof is not None, "Complete the Spyre scratchpad roofline calculation."
assert spyre_lpddr5_roof is not None, "Complete the Spyre LPDDR5 roofline calculation."
assert "TODO" not in dual_roofline_analysis, "Complete the dual roofline analysis (replace TODO)."
assert len(dual_roofline_analysis.strip()) > 100, "Analysis should address all 5 questions."

print("[OK] Self-check 5 passed!")
print("  Dual roofline analysis complete.")

### Prefill vs. Decode: Separate Roofline Analysis

LLM inference has two distinct phases with very different computational profiles:

| Phase | Characteristic | Arithmetic Intensity | Roofline Position |
|---|---|---|---|
| **Prefill** | Process full prompt in parallel | High (compute-bound) | Near/above ridge point |
| **Decode** | Generate one token at a time | Low (memory-bound) | Left of ridge point |

**Why separate them?**
- Prefill: large batch of tokens → high reuse of weights → compute-bound
- Decode: single token per step → each weight loaded once per token → memory-bound
- A single "inference AI" number hides this critical distinction

In [ ]:
# Prefill vs Decode roofline analysis
import json, os

# Load hardware profile
hw_path = os.path.join("..", "..", "corrected", "support", "hardware_profiles.json")
if not os.path.exists(hw_path):
    hw_path = os.path.join("..", "corrected", "support", "hardware_profiles.json")

# Model parameters (GPT-2 small as running example)
d_model = 768
n_heads = 12
n_layers = 12
d_head = d_model // n_heads
vocab_size = 50257
seq_len = 512  # prompt length
dtype_bytes = 2  # FP16

# === PREFILL phase ===
# FLOPs for one transformer layer (attention + FFM), full sequence
# Attention: 4 * seq_len * d_model^2 (Q,K,V,O projections) + 2 * seq_len^2 * d_model (attn scores + weighted sum)
# FFN: 2 * seq_len * d_model * 4*d_model (two linear layers)
attn_proj_flops = 4 * seq_len * d_model * d_model * 2  # *2 for FMA=2FLOPs
attn_score_flops = 2 * seq_len * seq_len * d_model * 2
ffn_flops = 2 * seq_len * d_model * (4 * d_model) * 2

prefill_flops_per_layer = attn_proj_flops + attn_score_flops + ffn_flops
prefill_flops_total = prefill_flops_per_layer * n_layers

# Bytes: weights loaded once for full sequence
params_per_layer = 4 * d_model**2 + 2 * d_model * (4*d_model)  # attn + ffn
prefill_bytes = params_per_layer * n_layers * dtype_bytes

prefill_ai = prefill_flops_total / prefill_bytes

# === DECODE phase (single token) ===
decode_seq_len = 1
attn_proj_flops_dec = 4 * decode_seq_len * d_model * d_model * 2
attn_score_flops_dec = 2 * decode_seq_len * seq_len * d_model * 2  # attend to full KV cache
ffn_flops_dec = 2 * decode_seq_len * d_model * (4 * d_model) * 2

decode_flops_per_layer = attn_proj_flops_dec + attn_score_flops_dec + ffn_flops_dec
decode_flops_total = decode_flops_per_layer * n_layers

# Bytes: same weights loaded but for single token
# Plus KV cache reads: 2 * n_layers * 2 * seq_len * d_model * dtype_bytes
kv_cache_bytes_read = 2 * n_layers * seq_len * d_model * dtype_bytes
decode_bytes = params_per_layer * n_layers * dtype_bytes + kv_cache_bytes_read

decode_ai = decode_flops_total / decode_bytes

print(f"{'Phase':<10} {'FLOPs':>12} {'Bytes':>12} {'AI (FLOPs/B)':>14} {'Regime'}")
print(f"{'─'*10} {'─'*12} {'─'*12} {'─'*14} {'─'*15}")
print(f"{'Prefill':<10} {prefill_flops_total:>12,.0f} {prefill_bytes:>12,.0f} {prefill_ai:>14.1f} {'compute-bound' if prefill_ai > 200 else 'memory-bound'}")
print(f"{'Decode':<10} {decode_flops_total:>12,.0f} {decode_bytes:>12,.0f} {decode_ai:>14.1f} {'compute-bound' if decode_ai > 200 else 'memory-bound'}")

# Ridge point comparison (T4 default)
ridge_point = 203  # T4 FP16: 65 TFLOPS / 320 GB/s
print(f"\nRidge point (T4 FP16): {ridge_point} FLOPs/byte")
print(f"  Prefill AI ({prefill_ai:.1f}) {'>' if prefill_ai > ridge_point else '<'} ridge point → {'COMPUTE-bound ✓' if prefill_ai > ridge_point else 'MEMORY-bound'}")
print(f"  Decode AI  ({decode_ai:.1f}) {'>' if decode_ai > ridge_point else '<'} ridge point → {'COMPUTE-bound' if decode_ai > ridge_point else 'MEMORY-bound ✓'}")

---

## Part 6 - Update the Accelerator Landscape Map

Add this module's row to the shared map you have been building since Lab 1.


In [ ]:
# Part 6 -- Landscape Map row: the roofline you measured this module.

# TODO: Complete this technical task using the instructions above.
# Try it yourself first. Open the Hint below if you need the full reference implementation.

# Your code here


<details>
<summary><strong>Hint</strong></summary>

If you need help, compare your work with the full reference implementation below.

```python
# Part 6 -- Landscape Map row: the roofline you measured this module.
landscape_row_w6 = {
    "accelerator":          gpu_name,
    "achieved_TFLOPS":      round(achieved_compute, 1),
    "achieved_BW_TB_s":     round(achieved_bw, 3),
    "empirical_ridge":      round(achieved_ridge, 0),
    "decode_FFN_AI":        round(ai_gemm_small, 2),
    "decode_bound":         ("memory" if ai_gemm_small < achieved_ridge else "compute"),
    "spyre_scratchpad_win": "when the working set fits in 64 MB (capacity-gated)",
}
for _k, _v in landscape_row_w6.items():
    print(f"  {_k:22s}: {_v}")
```

</details>


---
## Summary

In this lab you:
1. **Calculated** arithmetic intensity from first principles for prefill and decode operators
2. **Constructed** an empirical roofline for your GPU using measured peaks
3. **Profiled** real LLM kernels and placed them on the roofline
4. **Diagnosed** performance bottlenecks and proposed optimizations
5. **Compared** GPU and Spyre using dual roofline analysis

### Key Takeaways
- LLM decode is deeply memory-bound on all current GPUs (AI ~1 vs. ridge ~150-295)
- The same operation can be compute-bound or memory-bound depending on batch size
- Dataflow accelerators with on-chip scratchpad can shift the roofline for workloads that fit
- The dual roofline framework identifies the architecture crossover point quantitatively

---

## Optional: Spyre Deep Dive

For students with access to IBM Spyre documentation or the `torch-spyre` package:
- Calculate the exact tiling strategy needed to fit one transformer layer's weights in the 64 MB scratchpad
- Determine how many layers can be pipelined across Spyre's 32 cores
- Estimate the latency improvement for autoregressive decode on a 125M parameter model

> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.
